In [1]:
import numpy as np
import pandas as pd
from google.cloud import bigquery

pd.set_option('display.max_rows', None)

client = bigquery.Client()
project = client.project
datasets = ['bronze', 'silver', 'gold']


In [2]:
for ds in datasets:
    schema_query = f"""
        SELECT table_name, column_name
        FROM `{project}.{ds}.INFORMATION_SCHEMA.COLUMNS`
        WHERE table_name NOT LIKE '%_view%'
          AND data_type NOT LIKE 'ARRAY%'
          AND data_type NOT LIKE 'STRUCT%'
    """
    schema_df = client.query(schema_query).to_dataframe()

    df_dataset_results = pd.DataFrame()

    for table, group in schema_df.groupby('table_name'):
        cols = group['column_name'].tolist()
        null_counts = [f"COUNTIF(`{col}` IS NULL) AS `{col}`" for col in cols]

        q = f"""
        SELECT '{table}' AS table_name, COUNT(*) AS row_count,
        {', '.join(null_counts)}
        FROM `{project}.{ds}.{table}`
        """

        try:
            df_res = client.query(q).to_dataframe()
            df_melted = df_res.melt(id_vars=['table_name', 'row_count'], var_name='column_name', value_name='null_count')
            df_dataset_results = pd.concat([df_dataset_results, df_melted], ignore_index=True)
        except Exception as e:
            pass

    if not df_dataset_results.empty:
        df_dataset_results['null_percentage'] = np.where(
            df_dataset_results['row_count'] == 0,
            0.0,
            (df_dataset_results['null_count'] / df_dataset_results['row_count']) * 100
        )
        df_dataset_results['null_percentage'] = df_dataset_results['null_percentage'].round(2)

        df_dataset_results = df_dataset_results.sort_values(
            by=['null_percentage', 'null_count'],
            ascending=[False, False]
        ).reset_index(drop=True)

        print(f"NULL DATA REPORT - LAYER: {ds.upper()}")

        display(df_dataset_results)
    else:
        print(f"\nNo data to analyze in layer {ds}")

NULL DATA REPORT - LAYER: BRONZE


,table_name,row_count,column_name,null_count,null_percentage
0,raw_planner_BakeryOrders,5743,PlatformDiscountId,5743,100.00
1,raw_wallet_PaymentMethods,187,CardBrand,187,100.00
2,raw_wallet_RefundCreationErrors,122,StatusCode,122,100.00
3,raw_wallet_RefundCreationErrors,122,CodeLiteral,122,100.00
4,raw_catalog_ProductAttribute,3,BakeryProductId,3,100.00
5,raw_customer_manager_Consents,3,ExpiryDate,3,100.00
6,raw_wallet_Payments,2235,Error,2234,99.96
7,raw_wallet_Refunds,1063,ClaimId,1059,99.62
8,raw_wallet_Payments,2235,DiscountedAmount,2168,97.00
9,raw_wallet_Payments,2235,DiscountId,2167,96.96


NULL DATA REPORT - LAYER: SILVER


,table_name,row_count,column_name,null_count,null_percentage
0,stg_payment,2235,error,2234,99.96
1,stg_payment,2235,discounted_amount,2168,97.00
2,stg_payment,2235,id_discount,2167,96.96
3,stg_discount,21,id_customer,19,90.48
4,stg_delivery_items,1741,packed_quantity,1552,89.14
5,stg_bakery_offer,145,next_version_id,121,83.45
6,stg_payment,2235,id_client,1681,75.21
7,stg_refund,1063,base_amount,767,72.15
8,stg_bakery_offer,145,previous_version_offer_id,78,53.79
9,stg_bakery_product,83,id_previous_version_product,41,49.40


NULL DATA REPORT - LAYER: GOLD


,table_name,row_count,column_name,null_count,null_percentage
0,bridge_location_site,13,sk_location_site,0,0.0
1,bridge_location_site,13,id_location_site,0,0.0
2,bridge_location_site,13,id_location,0,0.0
3,bridge_location_site,13,id_site,0,0.0
4,bridge_offer_product,151,sk_offer_product,0,0.0
5,bridge_offer_product,151,id_offer_product,0,0.0
6,bridge_offer_product,151,id_offer,0,0.0
7,bridge_offer_product,151,id_product,0,0.0
8,bridge_offer_product,151,product_count,0,0.0
9,bridge_offer_product,151,product_in_offer_count,0,0.0


In [3]:
table_lineage = {
    'fact_payment': {'silver': 'stg_payment', 'bronze': 'raw_wallet_Payments'},
    'fact_order_item': {'silver': 'stg_bakery_order_items', 'bronze': 'raw_planner_BakeryOrderItems'},
    'fact_delivery_item': {'silver': 'stg_delivery_items', 'bronze': 'raw_delivery_manager_DeliveryItems'},
    'fact_refund_item': {'silver': 'stg_refund_items', 'bronze': 'raw_wallet_RefundItems'},
    'fact_planner_item': {'silver': 'stg_planner_item', 'bronze': 'raw_planner_PlannerItems'},

    'dim_customer': {'silver': 'stg_customer', 'bronze': 'raw_customer_manager_Customer'},
    'dim_customer_address': {'silver': 'stg_customer_address', 'bronze': 'raw_customer_manager_CustomerAddress'},
    'dim_product': {'silver': 'stg_bakery_product', 'bronze': 'raw_catalog_BakeryProduct'},
    'dim_payment_method': {'silver': 'stg_payment_method', 'bronze': 'raw_wallet_PaymentMethods'},
    'dim_site': {'silver': 'stg_site', 'bronze': 'raw_client_manager_Site'},
    'dim_delivery_man': {'silver': 'stg_delivery_man', 'bronze': 'raw_delivery_account_manager_DeliveryMan'},
    'dim_discount': {'silver': 'stg_discount', 'bronze': 'raw_wallet_Discounts'},
    'dim_location': {'silver': 'stg_location', 'bronze': 'raw_client_manager_Location'},
    'dim_allergen': {'silver': 'stg_allergen', 'bronze': 'raw_catalog_Allergen'},
    'dim_offer': {'silver': 'stg_bakery_offer', 'bronze': 'raw_catalog_BakeryOffer'},
    'dim_competitor': {'silver': 'stg_google_competitors', 'bronze': 'raw_google_competitors'}
}

print(f"Scanning {len(table_lineage)} main data paths. This will take a moment...")
dfs = []

for gold_table, lineage in table_lineage.items():
    silver_table = lineage['silver']
    bronze_table = lineage['bronze']

    targets = [
        ('1_Bronze', 'bronze', bronze_table),
        ('2_Silver', 'silver', silver_table),
        ('3_Gold', 'gold', gold_table)
    ]

    for layer_alias, dataset_name, table_name in targets:
        q = f"SELECT '{gold_table}' AS main_table, '{layer_alias}' AS layer, COUNT(*) AS row_count FROM `{project}.{dataset_name}.{table_name}`"

        try:
            dfs.append(client.query(q).to_dataframe())
        except Exception as e:
            print(f"Error for {dataset_name}.{table_name}: {e}")

if dfs:
    df_raw = pd.concat(dfs, ignore_index=True)
    df_pivot = df_raw.pivot(index='main_table', columns='layer', values='row_count')
    df_pivot.columns = [col.split('_')[1] for col in df_pivot.columns]

    print("\nDATA PATH CONSISTENCY AUDIT")
    display(df_pivot)
else:
    print("No data to display.")

Scanning 16 main data paths. This will take a moment...

DATA PATH CONSISTENCY AUDIT


,Bronze,Silver,Gold
main_table,,,
dim_allergen,15,15,16
dim_competitor,483,483,481
dim_customer,176,176,177
dim_customer_address,544,544,545
dim_delivery_man,6,6,7
dim_discount,21,21,22
dim_location,35,35,36
dim_offer,145,145,146
dim_payment_method,187,187,188


In [4]:
dataset = 'gold'

schema_query = f"""
    SELECT table_name, column_name
    FROM `{project}.{dataset}.INFORMATION_SCHEMA.COLUMNS`
    WHERE (table_name LIKE 'fact_%' OR table_name LIKE 'bridge_%')
      AND column_name LIKE 'id_%'
"""
schema_df = client.query(schema_query).to_dataframe()

queries = []
for table, group in schema_df.groupby('table_name'):
    cols = group['column_name'].tolist()

    orphan_counts = [f"COUNTIF(CAST(`{col}` AS STRING) = '-999999') AS `{col}`" for col in cols]

    q = f"""
    SELECT '{table}' AS table_name, COUNT(*) AS row_count,
    {', '.join(orphan_counts)}
    FROM `{project}.{dataset}.{table}`
    """
    queries.append(q)

print(f"Scanning {len(queries)} fact tables for orphaned records (-999999)...")

dfs = []
for q in queries:
    try:
        df_res = client.query(q).to_dataframe()

        df_melted = df_res.melt(id_vars=['table_name', 'row_count'], var_name='foreign_key', value_name='orphan_count')
        dfs.append(df_melted)
    except Exception as e:
        print(f"Error for table: {e}")

if dfs:
    df_orphans = pd.concat(dfs, ignore_index=True)

    df_orphans = df_orphans[df_orphans['orphan_count'] > 0].copy()

    if not df_orphans.empty:
        df_orphans['orphan_percentage'] = ((df_orphans['orphan_count'] / df_orphans['row_count']) * 100).round(2)
        df_orphans = df_orphans.sort_values(by=['orphan_percentage', 'orphan_count'], ascending=[False, False]).reset_index(drop=True)

        print(" BUSINESS EXCEPTION REPORT (Keys mapped to -999999)")
        display(df_orphans)
    else:
        print("\n Great news! The system works perfectly. No orphans (-999999) found in any fact tables.")
else:
    print("No data to scan.")

Scanning 9 fact tables for orphaned records (-999999)...
 BUSINESS EXCEPTION REPORT (Keys mapped to -999999)


,table_name,row_count,foreign_key,orphan_count,orphan_percentage
0,fact_refund_item,1132,id_claim,1131,99.91
1,fact_payment,2235,id_discount,2167,96.96
2,fact_payment,2235,id_client,1681,75.21
3,fact_refund_item,1132,id_client,844,74.56
4,fact_payment,2235,id_delivery,974,43.58
5,fact_refund_item,1132,id_location,354,31.27
6,fact_payment,2235,id_location,678,30.34
7,fact_payment,2235,id_site,403,18.03
8,fact_refund_item,1132,id_site,92,8.13
9,fact_delivery_item,1741,id_delivery_man,103,5.92
